# 파싱된 PDF 데이터로 LLM 질의 테스트

`document_intelligence_test.ipynb`에서 만든 `data/parsed/document_intelligence/<파일명>/content.md`(마크다운 변환 결과)를 컨텍스트로 Azure OpenAI에 던져서 질문/답변을 테스트하는 노트북입니다.

## 사전 준비
1. `document_intelligence_test.ipynb`를 먼저 실행해서 `data/parsed/document_intelligence/` 아래에 문서별 폴더(`raw.json`/`content.md`/`metadata.json`/원본 PDF)가 생성되어 있어야 합니다.
2. 프로젝트 루트(`azure-doc-ai-service/`)의 `.env`에 Azure OpenAI 값이 채워져 있어야 합니다 (`.env.example` 참고).

```
AZURE_OPENAI_ENDPOINT=https://<your-resource-name>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-api-key>
AZURE_OPENAI_DEPLOYMENT_NAME=<deployment-name>
AZURE_OPENAI_API_VERSION=2024-12-01-preview
```

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AOAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AOAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]
AOAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")

PARSED_DIR = Path.cwd().parent / "data" / "parsed" / "document_intelligence"

print("ENDPOINT:", AOAI_ENDPOINT)
print("DEPLOYMENT:", AOAI_DEPLOYMENT)
print("PARSED_DIR:", PARSED_DIR)

## 1. 파싱된 문서 목록 확인

`PARSED_DIR` 아래 폴더(=파일별 결과)를 스캔해서 `content.md`가 있는 것만 사용 가능한 문서로 나열합니다. 여기서 출력되는 `pdf_path`(원본 PDF 파일의 정확한 경로)를 아래 3~5절의 `pdf_path` 인자에 그대로 복사해서 사용하세요.

In [ ]:
def list_parsed_docs() -> list[dict]:
    docs = []
    if not PARSED_DIR.exists():
        return docs
    for folder in sorted(PARSED_DIR.iterdir()):
        md_path = folder / "content.md"
        if not folder.is_dir() or not md_path.exists():
            continue
        meta_path = folder / "metadata.json"
        meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
        pdf_candidates = sorted(folder.glob("*.pdf"))
        docs.append(
            {
                "name": folder.name,
                "filename": meta.get("filename", f"{folder.name}.pdf"),
                "pdf_path": pdf_candidates[0] if pdf_candidates else None,
                "content_md": md_path,
                "raw_json": folder / "raw.json",
                "content_chars": meta.get("content_chars"),
                "pages": meta.get("pages_analyzed"),
            }
        )
    return docs


available_docs = list_parsed_docs()
print(f"사용 가능한 문서 {len(available_docs)}개")
for d in available_docs:
    print(f"- {d['pdf_path']}  (pages={d['pages']}, chars={d['content_chars']})")

## 2. Azure OpenAI 클라이언트 준비

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_API_KEY,
    api_version=AOAI_API_VERSION,
)

## 3. 단일 문서 기준 질문/답변

원본 PDF 파일의 정확한 경로(`pdf_path`, 1절 출력 결과를 그대로 복사)를 지정하면 그 문서의 `content.md` 전체를 컨텍스트로 넣고 질문합니다. 잘라내지 않고 그대로 보내므로, 컨텍스트 길이 제한 에러(400/토큰 초과 등)가 나면 그때 잘라내는 로직을 추가하세요.

In [ ]:
def find_doc(pdf_path) -> dict:
    """원본 PDF 파일의 정확한 경로를 받아 해당 파싱 결과 폴더의 문서를 찾는다.

    document_intelligence_test.ipynb가 만든 폴더(`data/parsed/document_intelligence/<폴더명>/`)에는
    raw.json/content.md/metadata.json과 함께 원본 PDF가 들어있다. 그 PDF 파일의 경로를 그대로 받아서
    부모 폴더를 파싱 결과 폴더로 사용한다 (1절에서 출력되는 pdf_path를 그대로 복사해서 쓰면 된다).
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")

    folder = pdf_path.parent
    md_path = folder / "content.md"
    if not md_path.exists():
        raise FileNotFoundError(
            f"'{folder}'에 content.md가 없습니다. document_intelligence_test.ipynb를 먼저 실행하세요."
        )

    meta_path = folder / "metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    return {
        "name": folder.name,
        "filename": meta.get("filename", pdf_path.name),
        "pdf_path": pdf_path,
        "content_md": md_path,
        "raw_json": folder / "raw.json",
        "content_chars": meta.get("content_chars"),
        "pages": meta.get("pages_analyzed"),
    }


def ask(question: str, pdf_path) -> str:
    doc = find_doc(pdf_path)
    content = doc["content_md"].read_text(encoding="utf-8")

    system_prompt = (
        "당신은 주어진 문서 내용만 근거로 질문에 답하는 어시스턴트입니다. "
        "문서에 없는 내용은 추측하지 말고 '문서에서 확인할 수 없습니다'라고 답하세요."
    )
    user_prompt = f"[문서: {doc['name']}]\n---\n{content}\n---\n\n질문: {question}"

    response = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content or ""

In [ ]:
# 예시: 표가 깨졌던 파일로 테스트했던 그 문서에 질문해보기 (pdf_path는 1절 출력 결과에서 복사)
answer = ask(
    question="이 약관에서 위치정보 제공 동의 철회는 어떻게 하나요?",
    pdf_path=PARSED_DIR / "위치기반서비스+이용약관(별표)_20260331_V5.8" / "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf",
)
print(answer)

## 4. 답변 근거(evidence) 검증 - 원문과 정확히 일치하는지 + 몇 페이지인지 확인

LLM은 답변을 새로 생성하기 때문에 문서 문장을 그대로 베끼지 않고 의역/요약하는 경우가 많습니다. 이를 보완하기 위해:

1. 모델에게 `answer`(자연어 답변)와 `evidence`(근거 문장 — 문서 원문 그대로 인용)를 JSON으로 분리해서 받고,
2. Python 코드로 `evidence`의 각 문장이 실제 `content.md`에 **정확한 substring으로 존재하는지** 검증하고,
3. 발견된 위치(offset)가 `raw.json`의 `pages[].spans` 구간 중 어디에 속하는지로 **몇 페이지인지**도 계산합니다.

즉 "모델의 말"이 아니라 "코드가 원문에서 확인한 것"만 근거로 보여줍니다. ✅는 원문과 100% 일치, 〜는 공백/줄바꿈만 다름(내용은 일치), ❌는 문서에서 찾을 수 없음(모델이 지어냈을 가능성/환각 의심)을 뜻합니다.

Document Intelligence는 전체 텍스트(`content`, 곧 `content.md`와 동일)를 하나의 문자열로 두고, 각 페이지가 그 문자열의 어느 offset 구간([start, end))을 차지하는지를 `pages[].spans`로 기록합니다. evidence 문장이 `content.md`에서 발견된 offset이 어느 페이지 구간에 속하는지 찾으면 페이지 번호를 알 수 있습니다. (`raw.json`이 없으면 페이지 번호는 계산할 수 없습니다.)

### 4-1. evidence quote 보기 좋게 표시하기

`content.md`는 Document Intelligence가 `output_content_format=markdown`으로 만든 결과입니다. Document
Intelligence 공식 문서([Markdown elements](https://learn.microsoft.com/azure/ai-services/document-intelligence/concept/markdown-elements))에 따르면,
**표는 항상 HTML(`<table><tr><th>...`) 문법으로 나오고 마크다운 표 문법(`| a | b |`)은 쓰지 않습니다** (병합 셀 유무와 무관하게 전부 HTML). evidence는 이
`content.md`와 **정확히 일치해야** 검증(offset 계산, 페이지 매칭)이 되므로, 검증에 쓰는 quote 자체는 그 HTML을 그대로 유지해야 합니다.

대신 **화면에 보여줄 때만** 보기 좋게 가공하는 `render_quote_for_display()`를 추가합니다:

- quote 안에 `<table>` HTML이 통째로 들어있으면 태그는 버리고 셀 텍스트만 행/열로 복원해서 정렬된 표로 보여줍니다 — 약관의 표를 보는 것과 거의 같은 느낌입니다.
- 마크다운 표 문법(`| ... | ... |`, 구분선 `---`)도 같은 방식으로 처리하는 코드가 있지만, 이건 실제 Document Intelligence 출력에서는
  나오지 않는 형태라 평소엔 쓰일 일이 없는 방어 코드입니다 (LLM이 evidence를 원문 그대로 복사하지 않고 임의로 재구성하는 경우를 대비).
- 그 외 일반 문장은 남은 HTML 태그/주석/엔티티만 지우고 공백만 정리해서, 약관 원문 문장 그대로에 가깝게 보여줍니다.

검증 로직(`exact_match`, `pages` 계산 등)은 항상 원본 quote를 쓰고, 이 함수는 **출력 전용**입니다.

In [ ]:
import re
from html import unescape
from html.parser import HTMLParser

_TAG_RE = re.compile(r"<[^>]+>")
_TABLE_TAG_RE = re.compile(r"<(table|tr|td|th)\b", re.IGNORECASE)
_MD_TABLE_SEP_RE = re.compile(r"^\s*\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)+\|?\s*$")


class _HtmlTableGridParser(HTMLParser):
    """quote 안에 표 관련 HTML 태그(<table>/<tr>/<td>/<th>)가 들어있을 때 셀 텍스트만 행/열 grid로 복원한다.

    evidence quote는 표 전체가 아니라 <table> 여는 태그 없이 <tr>/<td> 일부만 잘려서 오는 경우가 많아서,
    <tr> 래핑이 없는 조각(예: <td>만 나열된 경우)도 하나의 행으로 처리하고, 닫는 태그 없이 끝나도 마지막 행을 흘리지 않는다.
    각 행이 <th>로 이루어졌는지(header_flags)도 같이 기록해서, quote 자체에 진짜 헤더가 포함됐는지 구분한다.

    colspan은 그 값을 span한 칸 수만큼 반복해서 채워 행마다 칸 수가 달라지는 것을 막는다.
    rowspan은 셀을 dict(컬럼 위치 -> 값)로 채워나가면서, span이 남은 컬럼을 다음 행 시작 시 미리
    채워두는 방식(pending)으로 처리한다 — rowspan이 "시작되는" 행이 quote에 포함돼 있어야만
    정확히 복원되고, quote가 그 행 없이 중간부터 잘려 있으면(정보 자체가 없으므로) 그 칸은 비어서 나온다.
    """

    def __init__(self):
        super().__init__()
        self.rows = []
        self.header_flags = []
        self._row = None  # col_index -> text
        self._row_has_th = False
        self._chunks = None
        self._in_cell = False
        self._colspan = 1
        self._rowspan = 1
        self._next_col = 0
        self._pending = {}  # col_index -> (text, remaining_rows)
        self._next_pending = {}

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self._row = {}
            self._row_has_th = False
            self._next_col = 0
            # 이전 행에서 넘어온(rowspan으로 아직 유효한) 셀을 이 행에 먼저 채워둔다.
            for col, (text, _remaining) in self._pending.items():
                self._row[col] = text
            self._next_pending = {
                col: (text, remaining - 1)
                for col, (text, remaining) in self._pending.items()
                if remaining - 1 > 0
            }
        elif tag in ("td", "th"):
            if self._row is None:
                self._row = {}
                self._row_has_th = False
                self._next_col = 0
                self._next_pending = {}
            if tag == "th":
                self._row_has_th = True
            attrs_dict = dict(attrs)
            try:
                self._colspan = max(1, int(attrs_dict.get("colspan", 1)))
            except (TypeError, ValueError):
                self._colspan = 1
            try:
                self._rowspan = max(1, int(attrs_dict.get("rowspan", 1)))
            except (TypeError, ValueError):
                self._rowspan = 1
            self._in_cell = True
            self._chunks = []

    def handle_endtag(self, tag):
        if tag == "tr" and self._row is not None:
            n_cols = max(self._row.keys()) + 1 if self._row else 0
            self.rows.append([self._row.get(i, "") for i in range(n_cols)])
            self.header_flags.append(self._row_has_th)
            self._pending = self._next_pending
            self._row = None
        elif tag in ("td", "th") and self._in_cell:
            text = " ".join("".join(self._chunks).split())
            while self._next_col in self._row:
                self._next_col += 1
            for i in range(self._colspan):
                self._row[self._next_col + i] = text
            if self._rowspan > 1:
                for i in range(self._colspan):
                    self._next_pending[self._next_col + i] = (text, self._rowspan - 1)
            self._next_col += self._colspan
            self._in_cell = False
            self._chunks = None
            self._colspan = 1
            self._rowspan = 1

    def handle_data(self, data):
        if self._in_cell:
            self._chunks.append(data)

    def close(self):
        super().close()
        if self._row:
            n_cols = max(self._row.keys()) + 1 if self._row else 0
            self.rows.append([self._row.get(i, "") for i in range(n_cols)])
            self.header_flags.append(self._row_has_th)
            self._row = None


def _quote_table_rows(quote: str) -> tuple[list[list[str]] | None, bool]:
    """quote가 표(HTML 표 조각 또는 마크다운 표)면 (행/열 grid, quote 자체에 진짜 헤더가 있었는지)를 반환한다.

    표가 아니면 (None, False). Document Intelligence 공식 문서(Markdown elements)에 따르면 표는 병합 셀
    유무와 상관없이 항상 HTML(<table>/<tr>/<th>/<td>, colspan/rowspan 속성 포함)로만 나오고 마크다운 표
    문법(| a | b |)은 쓰지 않는다 — 그래서 실제 evidence quote는 거의 항상 HTML 분기를 탄다. 마크다운
    파이프 표 인식 분기는 evidence quote가 (LLM이 원문 그대로 복사하지 않고 재구성하는 등의 이유로)
    HTML이 아닌 형태로 올 경우를 대비한 방어 코드일 뿐, 실제 Document Intelligence 출력을 반영한 게 아니다.
    HTML/마크다운 모두 데이터 행 1개짜리 조각(헤더 없이 한 행만 인용된 경우)도 표로 인식한다.
    "진짜 헤더 있음"은 HTML의 경우 첫 행이 <th>로 이루어졌을 때, 마크다운 표의 경우 첫 행 바로 다음에
    구분선(---)이 있을 때만 확정한다 — 그 외에는 첫 행이 헤더인지 데이터인지 알 수 없으므로 함부로 헤더로 취급하지 않는다.
    """
    if _TABLE_TAG_RE.search(quote):
        parser = _HtmlTableGridParser()
        parser.feed(quote)
        parser.close()
        if not parser.rows:
            return None, False
        has_own_header = bool(parser.header_flags and parser.header_flags[0])
        return parser.rows, has_own_header

    raw_lines = [ln for ln in quote.splitlines() if ln.strip()]
    if not raw_lines:
        return None, False
    has_own_header = len(raw_lines) > 1 and bool(_MD_TABLE_SEP_RE.match(raw_lines[1]))
    lines = [ln for ln in raw_lines if not _MD_TABLE_SEP_RE.match(ln)]
    # 마크다운 표 행은 관례상 "|"로 시작한다 — 이 조건으로 우연히 "|"가 섞인 일반 문장과 구분한다.
    if lines and all(ln.strip().startswith("|") for ln in lines):
        rows = [[c.strip() for c in ln.strip().strip("|").split("|")] for ln in lines]
        if rows and len(rows[0]) > 1 and len({len(r) for r in rows}) == 1:
            return rows, has_own_header
    return None, False


def _clean_plain_text(quote: str) -> str:
    """표가 아닌 일반 문장에서 남은 HTML 태그/주석/엔티티를 지우고 공백만 정리한다."""
    text = unescape(_TAG_RE.sub(" ", quote))
    text = "\n".join(ln.strip() for ln in text.splitlines())
    text = re.sub(r"\n{2,}", "\n", text).strip()
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text


def _prepend_header(rows: list[list[str]], header: list[str] | None) -> list[list[str]]:
    """복원된 표 grid 맨 앞에 원본 표의 헤더 행을 붙인다 (이미 같은 헤더가 있으면 중복으로 붙이지 않는다)."""
    if not header:
        return rows
    if rows and [c.strip() for c in rows[0]] == [c.strip() for c in header]:
        return rows
    return [header] + rows


def _rows_for_render(quote: str, header: list[str] | None) -> list[list[str]] | None:
    """render 함수 공통 로직: quote를 grid로 파싱하고, 헤더 행을 확보한다.

    quote 자체에 이미 확인된 헤더가 있으면(has_own_header) 그걸 그대로 쓰고 raw.json의 header는
    무시한다 — 텍스트가 완전히 똑같지 않으면 _prepend_header의 중복 검사를 통과하지 못해 헤더가
    두 번 나오는 문제가 있었다. quote 자체엔 헤더가 없을 때만 raw.json의 header를 붙이고,
    그마저도 없으면 데이터 행이 헤더로 오인되지 않도록 빈 헤더 행을 붙인다.
    """
    rows, has_own_header = _quote_table_rows(quote)
    if not rows:
        return None
    if has_own_header:
        return rows
    if header:
        return _prepend_header(rows, header)
    n_cols = max(len(r) for r in rows)
    return [[""] * n_cols] + rows


def _grid_to_lines(rows: list[list[str]]) -> list[str]:
    """grid를 터미널에서 보기 좋게 열 너비를 맞춘 텍스트 줄들로 변환한다."""
    n_cols = max((len(r) for r in rows), default=0)
    widths = [max((len(r[i]) for r in rows if i < len(r)), default=0) for i in range(n_cols)]
    lines = []
    for row in rows:
        padded = [(row[i] if i < len(row) else "").ljust(widths[i]) for i in range(n_cols)]
        lines.append(" | ".join(padded).rstrip())
    return lines


def _grid_to_markdown_table(rows: list[list[str]]) -> str:
    """grid를 Slack/이메일/문서에 붙여도 표로 렌더되는 마크다운 표 문법으로 변환한다. rows[0]은 항상 헤더로 취급된다."""
    n_cols = max((len(r) for r in rows), default=0)

    def esc(cell: str) -> str:
        return cell.replace("|", "\\|").replace("\n", " ")

    header = [esc(c) for c in rows[0]] + [""] * (n_cols - len(rows[0]))
    lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join(["---"] * n_cols) + " |",
    ]
    for row in rows[1:]:
        padded = [esc(c) for c in row] + [""] * (n_cols - len(row))
        lines.append("| " + " | ".join(padded) + " |")
    return "\n".join(lines)


def render_quote_for_display(quote: str, header: list[str] | None = None) -> str:
    """evidence quote(검증에는 그대로 쓰이는 원본 문자열)를 터미널 출력용으로만 보기 좋게 다듬는다.

    header를 주면(원본 표에서 복원한 헤더 행) quote 안에 헤더가 없어도 표 맨 위에 붙여서 보여준다.
    header도 없고 quote 자체에도 진짜 헤더가 없으면, 데이터 행이 헤더로 오인되지 않도록 빈 헤더 행을 붙인다.
    """
    rows = _rows_for_render(quote, header)
    if rows:
        return "\n".join(_grid_to_lines(rows))
    return _clean_plain_text(quote)


def render_quote_as_markdown(quote: str, header: list[str] | None = None) -> str:
    """evidence quote를 Slack/이메일/문서에 붙여넣을 마크다운으로 다듬는다 (표는 마크다운 표로, 문장은 정리된 텍스트로).

    header를 주면 마크다운 표의 헤더 행으로 사용한다. header도 없고 quote 자체에도 진짜 헤더가 없으면,
    데이터 행이 헤더 자리에 들어가 사라져 보이는 걸 막기 위해 빈 헤더 행을 붙인다.
    """
    rows = _rows_for_render(quote, header)
    if rows:
        return _grid_to_markdown_table(rows)
    return _clean_plain_text(quote)

In [ ]:
EVIDENCE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "answer_with_evidence",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "answer": {"type": "string"},
                "evidence": {
                    "type": "array",
                    "description": "답변의 근거가 되는 문서 원문 문장 (의역/요약 없이 원문 그대로 복사)",
                    "items": {"type": "string"},
                },
            },
            "required": ["answer", "evidence"],
            "additionalProperties": False,
        },
    },
}


def build_page_spans(doc: dict) -> list[tuple[int, int, int]]:
    """raw.json의 pages[].spans에서 (start_offset, end_offset, page_number) 목록을 만든다."""
    raw_path = doc["raw_json"]
    if not raw_path.exists():
        return []

    raw = json.loads(raw_path.read_text(encoding="utf-8"))
    page_spans = []
    for page in raw.get("pages", []):
        spans = page.get("spans") or []
        if not spans:
            continue
        start = min(s["offset"] for s in spans)
        end = max(s["offset"] + s["length"] for s in spans)
        page_spans.append((start, end, page["pageNumber"]))

    page_spans.sort(key=lambda x: x[0])
    return page_spans


def offset_to_page(offset: int, page_spans: list[tuple[int, int, int]]) -> int | None:
    for start, end, page_number in page_spans:
        if start <= offset < end:
            return page_number
    # 페이지 사이 구분자(<!-- PageBreak --> 등) 위치라 정확히 안 걸치는 경우, 가장 가까운 이전 페이지로 대체
    candidates = [p for p in page_spans if p[0] <= offset]
    return candidates[-1][2] if candidates else None


def find_pages_for_quote(quote: str, content: str, page_spans: list[tuple[int, int, int]]) -> list[int]:
    if not page_spans:
        return []
    pages = set()
    start_idx = 0
    while True:
        idx = content.find(quote, start_idx)
        if idx == -1:
            break
        page = offset_to_page(idx, page_spans)
        if page is not None:
            pages.add(page)
        start_idx = idx + 1
    return sorted(pages)


_raw_tables_cache: dict[str, list[dict]] = {}


def _load_raw_tables(doc: dict) -> list[dict]:
    """raw.json의 tables(표 구조 원본 데이터: 행/열 위치, 헤더 셀 표시 등)를 읽어온다. 문서당 한 번만 파싱해서 캐싱한다."""
    raw_path = doc.get("raw_json")
    if not raw_path or not raw_path.exists():
        return []
    key = str(raw_path)
    if key not in _raw_tables_cache:
        raw = json.loads(raw_path.read_text(encoding="utf-8"))
        _raw_tables_cache[key] = raw.get("tables", []) or []
    return _raw_tables_cache[key]


def _table_bounds(table: dict) -> tuple[int, int] | None:
    """표의 [start, end) offset 범위를 계산한다. 표 전체 spans가 비어있으면 셀들의 spans로 유추한다."""
    spans = table.get("spans") or [s for c in table.get("cells", []) or [] for s in (c.get("spans") or [])]
    if not spans:
        return None
    start = min(s["offset"] for s in spans)
    end = max(s["offset"] + s["length"] for s in spans)
    return start, end


def _build_header_row_from_cells(row_cells: list[dict], col_count: int) -> list[str]:
    """헤더 행에 해당하는 셀 목록을 실제 헤더 문자열 리스트로 조립한다. columnSpan만큼 값을 반복해서 채운다."""
    header = [""] * col_count
    for c in row_cells:
        col = c["columnIndex"]
        span = max(1, c.get("columnSpan", 1) or 1)
        text = " ".join((c.get("content") or "").split())
        for offset in range(span):
            if 0 <= col + offset < col_count:
                header[col + offset] = text
    return header


def _table_confirmed_header(table: dict) -> list[str] | None:
    """columnHeader로 명시된 셀이 있을 때만 헤더 행을 만든다 (추정 없이 확실한 경우만)."""
    cells = table.get("cells", []) or []
    header_cells = [c for c in cells if c.get("kind") == "columnHeader"]
    if not header_cells:
        return None
    header_row_index = min(c["rowIndex"] for c in header_cells)
    row_cells = [c for c in header_cells if c["rowIndex"] == header_row_index]
    col_count = table.get("columnCount") or max(
        c["columnIndex"] + max(1, c.get("columnSpan", 1) or 1) for c in row_cells
    )
    return _build_header_row_from_cells(row_cells, col_count)


def find_evidence_table_header(quote: str, content: str, doc: dict) -> list[str] | None:
    """evidence quote가 속한 표에 확인된(kind="columnHeader") 헤더가 있으면 그걸 반환하고, 없으면 None.

    표가 여러 페이지에 걸쳐 쪼개지는 경우, 이전 페이지의 표를 열 구조(columnIndex)로 추정해서 헤더를
    가져오는 방법도 만들어서 실제 KT 5G 약관 문서로 검증까지 했었다. 하지만 그건 결국 "구조가 비슷하니
    아마 같은 표일 것"이라는 추정일 뿐이라, 우연히 구조만 비슷한 무관한 표(헤더가 없는 표 바로 이전
    페이지에 마침 열 구조가 호환되는 다른 표가 있는 경우)와 잘못 연결될 위험이 있었다. "확인 안 된 걸
    확인된 것처럼 보여주지 않는다"는 원칙에 따라, 이 표 자체에 진짜 헤더가 없으면 다른 표에서 추정해오지
    않고 그냥 None을 반환한다 (화면에는 빈 헤더 행으로 표시되어, 데이터가 헤더로 둔갑하는 일은 없다).
    """
    idx = content.find(quote)
    if idx == -1:
        return None
    end_idx = idx + len(quote)

    for table in _load_raw_tables(doc):
        bounds = _table_bounds(table)
        if bounds and bounds[0] < end_idx and idx < bounds[1]:
            return _table_confirmed_header(table)
    return None


def ask_with_evidence(question: str, pdf_path) -> dict:
    doc = find_doc(pdf_path)
    content = doc["content_md"].read_text(encoding="utf-8")
    page_spans = build_page_spans(doc)

    system_prompt = (
        "당신은 주어진 문서 내용만 근거로 질문에 답하는 어시스턴트입니다. "
        "문서에 없는 내용은 추측하지 말고 '문서에서 확인할 수 없습니다'라고 답하세요. "
        "evidence 필드에는 답변의 근거가 되는 문장을 문서 원문 그대로(의역/요약/축약 없이, 띄어쓰기까지 그대로) 복사해서 넣으세요."
    )
    user_prompt = f"[문서: {doc['name']}]\n---\n{content}\n---\n\n질문: {question}"

    response = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=EVIDENCE_SCHEMA,
    )

    parsed = json.loads(response.choices[0].message.content or "{}")

    evidence = []
    for quote in parsed.get("evidence", []):
        exact_match = quote in content
        loosely_match = " ".join(quote.split()) in " ".join(content.split())
        pages = find_pages_for_quote(quote, content, page_spans) if exact_match else []
        table_header = find_evidence_table_header(quote, content, doc) if exact_match else None
        evidence.append(
            {
                "quote": quote,
                "exact_match": exact_match,
                "verified": exact_match or loosely_match,
                "pages": pages,
                "table_header": table_header,
            }
        )

    return {
        "doc": doc["name"],
        "question": question,
        "answer": parsed.get("answer", ""),
        "evidence": evidence,
    }


def print_result(result: dict) -> None:
    print(f"[문서: {result['doc']}]")
    print(f"질문: {result['question']}\n")
    print(f"답변:\n{result['answer']}\n")
    print("근거(evidence):")
    if not result["evidence"]:
        print("  (LLM이 근거 문장을 제시하지 않았습니다)")
    for i, ev in enumerate(result["evidence"], start=1):
        if ev["exact_match"]:
            mark = "✅ 원문과 정확히 일치"
        elif ev["verified"]:
            mark = "〜 내용은 일치 (공백/줄바꿈만 다름)"
        else:
            mark = "❌ 문서에서 찾을 수 없음 (환각 의심)"

        if ev["pages"]:
            page_label = ", ".join(str(p) for p in ev["pages"])
            mark += f" (page {page_label})"
        elif ev["exact_match"]:
            mark += " (페이지 확인 불가: raw.json 없음)"

        print(f"  {i}. {mark}")
        for line in render_quote_for_display(ev["quote"], ev.get("table_header")).splitlines():
            print(f"     │ {line}")

In [ ]:
result = ask_with_evidence(
    question="이 약관에서 위치정보 제공 동의 철회는 어떻게 하나요?",
    pdf_path=PARSED_DIR / "위치기반서비스+이용약관(별표)_20260331_V5.8" / "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf",
)
print_result(result)

## 5. 항목(items) vs 약관 원문 검증

아래 형태의 요청 데이터가 들어오면, 지정한 약관 PDF(들)의 원문과 대조해서 `items`의 각 항목(`itemNm`/`value`)이 그 약관에 실제로 있는 내용인지 판정합니다.

```json
{
  "request_id": "123",
  "objects": [
    {
      "name": "요고 69",
      "items": [
        {"value": "Y", "itemNm": "초이스 상품여부"},
        {"value": "개인, 미성년자, 외국인", "itemNm": "이용 가능 고객"}
      ]
    }
  ]
}
```

- `name`은 상품명일 수도, 다른 대상일 수도 있음 — `items`는 그 `name`에 대한 속성 목록으로 취급되어 약관과 대조됩니다.
- 문서 매칭은 3~4절과 동일하게 `find_doc()`으로 **원본 PDF 파일의 정확한 경로**를 받아 문서를 찾습니다. 경로는 1절에서 출력되는 `pdf_path`를 그대로 복사해서 쓰면 됩니다.
- 지금은 테스트 단계라 `name` → 대조할 PDF 매핑은 자동화하지 않고, `PDF_PATHS`에 **고정된 경로 목록**을 직접 적어서 그 문서들 각각에 대해 검증합니다 (문서별로 결과를 따로 보여줌).
- 판정은 4절의 evidence 검증과 동일하게: 모델이 `evidence`(약관 원문 그대로 인용)를 내면 → 코드가 `content.md`에서 실제 존재하는지/몇 페이지인지 확인 → **"일치"인데 근거가 원문에서 확인되지 않으면** 환각 의심 경고를 표시합니다.
- 판정 값: `일치` / `불일치` / `확인불가`.

In [ ]:
ITEM_VERIFICATION_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "item_verification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "results": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "itemNm": {"type": "string"},
                            "value": {"type": "string"},
                            "evidence": {
                                "type": "array",
                                "description": (
                                    "먼저 약관 본문에서 이 name(대상 명칭)에 해당하는 부분으로 범위를 좁힌 뒤, "
                                    "그 안에서 이 항목과 관련된 문장을 찾아 원문 그대로(의역/요약 없이) 복사. "
                                    "value가 여러 개의 값을 나열한 것이면 낱개 값별 근거를 모두 포함. 관련 내용이 전혀 없으면 빈 배열."
                                ),
                                "items": {"type": "string"},
                            },
                            "reason": {
                                "type": "string",
                                "description": (
                                    "위 evidence를 근거로 이 항목이 value와 맞는지/틀리는지/확인 불가능한지 설명. "
                                    "value에 낱개 값이 여러 개면 낱개 값별로 확인됨/모순/근거없음을 구체적으로 설명"
                                ),
                            },
                            "verdict": {
                                "type": "string",
                                "description": "evidence와 reason을 바탕으로 마지막에 결정하는 최종 판정",
                                "enum": ["일치", "불일치", "확인불가"],
                            },
                        },
                        "required": ["itemNm", "value", "evidence", "reason", "verdict"],
                        "additionalProperties": False,
                    },
                },
            },
            "required": ["results"],
            "additionalProperties": False,
        },
    },
}


def verify_items_against_doc(name: str, items: list[dict], pdf_path) -> dict:
    """items(itemNm/value 목록)이 pdf_path(원본 PDF의 정확한 경로)와 일치하는 약관 문서 원문과 일치하는지 검증한다.

    value가 없는 item은 대조할 값 자체가 없으므로 건너뛴다 (results에도 포함되지 않는다).
    """
    doc = find_doc(pdf_path)
    content = doc["content_md"].read_text(encoding="utf-8")
    page_spans = build_page_spans(doc)

    items = [it for it in items if "value" in it]
    if not items:
        return {"name": name, "doc": doc["name"], "results": []}

    items_text = "\n".join(f"- {it['itemNm']}: {it['value']}" for it in items)

    system_prompt = (
        "당신은 항목 정보(itemNm/value)가 실제 약관 원문과 일치하는지 검증하는 어시스턴트입니다. "
        "[대상 명칭](name)은 상품명일 수도, 이벤트명일 수도, 그 외 다른 대상일 수도 있습니다. "
        "약관 문서에는 여러 상품/이벤트/조건이 함께 나열되어 있을 수 있으니, "
        "반드시 먼저 문서 안에서 이 name에 해당하는 부분(표의 행, 절, 조항 등)을 특정하고, "
        "그 범위 안에서만 evidence를 찾으세요. name과 무관한 다른 상품/이벤트에 대한 문장을 근거로 삼지 마세요. "
        "약관이 표 형태로 되어 있다면 헤더 행과 실제 값의 열 대응이 이 name과 정확히 일치하는지 특히 주의해서 확인하세요 "
        "(마크다운 변환 과정에서 병합 셀이나 열이 어긋나 보일 수 있습니다). "
        "itemNm은 약관에 그대로 등장하지 않을 수 있습니다 — 정확히 같은 단어가 아니라 같은 의미를 가리키는 "
        "동의어/유사 표현이어도 근거로 인정하되, evidence 인용문 자체는 반드시 약관 원문 그대로(의역 없이, 띄어쓰기까지 그대로) 복사하세요. "
        "value가 하나의 단일 값이 아니라 여러 개의 대상·조건·항목을 나열한 것으로 보이면 "
        "(구분자가 콤마든 슬래시든 '및'이든 범위 표현이든 무엇이든 상관없이), 그 낱개 단위 각각을 약관에서 개별적으로 확인하세요. "
        "각 항목마다 반드시 이 순서로 작업하세요: "
        "(1) 위 기준(대상 특정, 표 열 대응 확인, 동의어 허용, 다중 값 분해)에 따라 약관 본문에서 관련 문장을 찾아 "
        "evidence에 원문 그대로 복사한다. 낱개 값마다 근거 위치가 다르면 각각 별도의 문장으로 추가한다. "
        "관련 내용을 전혀 찾을 수 없으면 evidence는 빈 배열로 둔다. "
        "(2) 그 evidence를 근거로 reason에 설명한다. value에 낱개 값이 여러 개면 각각에 대해 "
        "'약관에서 확인됨 / 약관과 모순됨 / 근거를 찾지 못함' 중 어디에 해당하는지 하나씩 짚어서 설명한다. "
        "(3) 마지막으로 evidence와 reason에서 실제로 뒷받침된 내용만 바탕으로 verdict를 정한다: "
        "value를 구성하는 모든 낱개 값이 각각 약관에서 명확히 뒷받침될 때만 '일치'로, "
        "그 중 하나라도 약관 내용과 명백히 모순되면(예: 약관이 그 값을 명시적으로 제외하거나 다르게 규정) 전체를 '불일치'로, "
        "모순되는 값은 없으나 일부 낱개 값의 근거를 찾지 못했다면 '확인불가'로 판정한다. "
        "value가 단일 값인 항목도 동일한 기준을 적용한다: evidence가 비어있거나 이 항목을 명확히 뒷받침하지 못하면 '확인불가', "
        "evidence가 value와 다른 내용을 명시하면 '불일치', evidence가 value와 일치하는 내용을 명확히 명시할 때만 '일치'로 판정한다. "
        "verdict가 evidence/reason과 모순되면 안 된다."
    )
    user_prompt = (
        f"[대상 명칭: {name}]\n"
        f"[대조할 약관: {doc['name']}]\n---\n{content}\n---\n\n"
        f"다음 항목들이 위 약관 내용과 일치하는지 각각 판정하세요:\n{items_text}"
    )

    response = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=ITEM_VERIFICATION_SCHEMA,
    )
    parsed = json.loads(response.choices[0].message.content or "{}")

    results = []
    for item in parsed.get("results", []):
        evidence = []
        for quote in item.get("evidence", []):
            exact_match = quote in content
            loosely_match = " ".join(quote.split()) in " ".join(content.split())
            pages = find_pages_for_quote(quote, content, page_spans) if exact_match else []
            table_header = find_evidence_table_header(quote, content, doc) if exact_match else None
            evidence.append(
                {
                    "quote": quote,
                    "exact_match": exact_match,
                    "verified": exact_match or loosely_match,
                    "pages": pages,
                    "table_header": table_header,
                }
            )
        results.append(
            {
                "itemNm": item.get("itemNm", ""),
                "value": item.get("value", ""),
                "evidence": evidence,
                "reason": item.get("reason", ""),
                "verdict": item.get("verdict", ""),
            }
        )

    return {"name": name, "doc": doc["name"], "results": results}


def verify_request(request: dict, pdf_paths: list) -> list[dict]:
    """request의 각 object(name+items)를 pdf_paths에 적힌 원본 PDF 경로마다 각각 검증한다."""
    all_results = []
    for obj in request.get("objects", []):
        name = obj.get("name", "")
        items = obj.get("items", [])
        for pdf_path in pdf_paths:
            all_results.append(verify_items_against_doc(name, items, pdf_path))
    return all_results


def print_verification(result: dict) -> None:
    print(f"[대상: {result['name']}] vs [약관: {result['doc']}]")
    for item in result["results"]:
        if item["verdict"] == "일치":
            mark = "✅ 일치"
        elif item["verdict"] == "불일치":
            mark = "❌ 불일치"
        else:
            mark = "➖ 확인불가"

        has_verified_evidence = any(e["verified"] for e in item["evidence"])
        if item["verdict"] == "일치" and item["evidence"] and not has_verified_evidence:
            mark += " ⚠️(근거 원문 미확인 - 환각 의심)"

        print(f"  - {item['itemNm']}: {item['value']}  → {mark}")
        print(f"    사유: {item['reason']}")
        for ev in item["evidence"]:
            if ev["exact_match"]:
                tag = "✅ 원문 일치"
            elif ev["verified"]:
                tag = "〜 내용 일치(공백만 차이)"
            else:
                tag = "❌ 원문에서 못 찾음"
            page_label = f" (page {', '.join(str(p) for p in ev['pages'])})" if ev["pages"] else ""
            print(f"      · {tag}{page_label}:")
            for line in render_quote_for_display(ev["quote"], ev.get("table_header")).splitlines():
                print(f"          {line}")
    print()

### 5-1. 테스트 실행

`PDF_PATHS`에 대조할 약관 원본 PDF의 정확한 경로를 채워 넣고 실행하세요 (1절에서 출력되는 `pdf_path`를 그대로 복사). `objects[].items`는 해당 `objects[].name`에 대한 속성 목록으로 취급되어, 각 PDF마다 한 번씩 검증됩니다.

In [ ]:
REQUEST_SAMPLE = {
    "request_id": "123",
    "objects": [
        {
            "name": "요고 69",
            "items": [
                {"value": "Y", "itemNm": "초이스 상품여부"},
                {"value": "개인, 미성년자, 외국인", "itemNm": "이용 가능 고객"},
                {"itemNm": "이용 금액", "description": "설명"},
            ],
        }
    ],
}

# 검증에 사용할 원본 PDF의 정확한 경로를 채워 넣으세요. (1절에서 출력되는 pdf_path를 그대로 복사)
# 예: PARSED_DIR / "위치기반서비스+이용약관(별표)_20260331_V5.8" / "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf"
PDF_PATHS = [
    # "여기에 원본 PDF의 정확한 경로",
]

results = verify_request(REQUEST_SAMPLE, PDF_PATHS)
for r in results:
    print_verification(r)

### 5-2. request와 같은 구조로 응답 만들기

`print_verification()`은 사람이 눈으로 보기 위한 출력이고, 실제로는 요청받은 JSON과 같은 구조로 `items`마다 `verdict`/`reason`/`evidence`만 채워서 돌려주면 됩니다. `build_response()`가 그 역할을 합니다 (PDF 1개 기준 — 여러 PDF와 대조하고 싶으면 PDF마다 한 번씩 호출).

In [ ]:
def build_response(request: dict, pdf_path) -> dict:
    """request와 동일한 구조(request_id/objects/name/items)를 유지한 채,
    각 item(itemNm/value)에 verdict/reason/evidence만 채워서 반환한다.
    value가 없는 item은 대조 대상이 아니므로 결과에서 제외한다."""
    response_objects = []
    objects = request.get("objects", [])
    total = len(objects)
    for idx, obj in enumerate(objects, start=1):
        name = obj.get("name", "")
        print(f"[{idx}/{total}] 처리 중: {name}")
        items = [it for it in obj.get("items", []) if "value" in it]
        verified = verify_items_against_doc(name, items, pdf_path)

        # itemNm+value 조합으로 원본 items 순서와 검증 결과를 매칭
        result_by_key = {(r["itemNm"], r["value"]): r for r in verified["results"]}

        new_items = []
        for it in items:
            key = (it.get("itemNm"), it.get("value"))
            r = result_by_key.get(key, {})
            new_items.append(
                {
                    "itemNm": it.get("itemNm"),
                    "value": it.get("value"),
                    "verdict": r.get("verdict", "확인불가"),
                    "reason": r.get("reason", ""),
                    "evidence": [
                        {
                            "quote": e["quote"],
                            "verified": e["verified"],
                            "pages": e["pages"],
                            "table_header": e.get("table_header"),
                        }
                        for e in r.get("evidence", [])
                    ],
                }
            )

        response_objects.append({"name": name, "items": new_items})

    return {
        "request_id": request.get("request_id"),
        "doc": verified["doc"] if response_objects else str(pdf_path),
        "objects": response_objects,
    }


# 예시 (5-1의 REQUEST_SAMPLE, PDF_PATHS[0] 사용)
if PDF_PATHS:
    response = build_response(REQUEST_SAMPLE, PDF_PATHS[0])
    print(json.dumps(response, ensure_ascii=False, indent=2))

### 5-3. 검증 결과를 Slack/이메일/문서용 리포트로 변환하기

`build_response()`가 만드는 JSON은 다른 시스템에 그대로 넘기기 위한 데이터 구조입니다. 이걸 Slack 메시지/이메일 본문/문서에 **그대로 붙여넣을 수 있는 마크다운 리포트**로 바꾸려면 `build_verification_report_markdown()`을 쓰세요.

- 표 형태 evidence는 진짜 마크다운 표(`| ... | ... |`)로, 일반 문장은 정리된 텍스트로 나옵니다.
- 판정(`일치`/`불일치`/`확인불가`)마다 이모지 배지를 붙이고, 근거마다 페이지 번호와 원문 확인 여부를 표시합니다.
- 반환값은 순수 문자열이라 Slack API의 `text`, 이메일 본문, `.md` 파일 저장 등에 그대로 넣으면 됩니다.

In [ ]:
_VERDICT_BADGE = {"일치": "✅ 일치", "불일치": "❌ 불일치"}

_MD_SPECIAL_RE = re.compile(r"([\\`*_\[\]|])")


def _md_escape(text) -> str:
    """표 셀/제목처럼 한 줄이어야 하는 마크다운 요소에 값을 안전하게 끼워 넣는다.

    개행은 공백으로 접어서 표 행이나 헤딩이 여러 줄로 쪼개져 마크다운이 깨지는 것을 막고,
    마크다운 특수문자(\\ ` * _ [ ] |)는 이스케이프해서 값에 우연히 섞인 문법이 표 구분자나
    굵게/이탤릭/코드/링크 서식으로 잘못 해석되어 뒤 내용까지 깨뜨리지 않게 한다.
    request로 들어오는 itemNm/value/name처럼 신뢰할 수 없는 입력을 마크다운에 넣을 때 쓴다.
    """
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = " ".join(text.split())
    return _MD_SPECIAL_RE.sub(r"\\\1", text)


def _blockquote(text: str) -> str:
    return "\n".join(f"> {line}" if line else ">" for line in text.splitlines())


def build_verification_report_markdown(response: dict) -> str:
    """build_response()의 출력을 Slack/이메일/문서에 그대로 붙여넣을 수 있는 마크다운 리포트로 변환한다."""
    lines = [f"## 검증 결과 — {_md_escape(response.get('doc', ''))}"]
    if response.get("request_id"):
        lines.append(f"_request_id: {_md_escape(response['request_id'])}_")

    for obj in response.get("objects", []):
        items = obj.get("items", [])
        lines += ["", f"### {_md_escape(obj.get('name', ''))}", "", "| 항목 | 값 | 판정 |", "| --- | --- | --- |"]
        for item in items:
            badge = _VERDICT_BADGE.get(item.get("verdict"), "➖ 확인불가")
            itemNm = _md_escape(item.get("itemNm", ""))
            value = _md_escape(item.get("value", ""))
            lines.append(f"| {itemNm} | {value} | {badge} |")

        for item in items:
            badge = _VERDICT_BADGE.get(item.get("verdict"), "➖ 확인불가")
            itemNm = _md_escape(item.get("itemNm", ""))
            value = _md_escape(item.get("value", ""))
            lines += ["", f"**{itemNm}** ({value}) — {badge}", ""]

            # 사유/근거 각각을 별도 '문단'으로 만들어서, 빈 ">" 줄로 서로 떨어뜨린다.
            # (인용 블록 안에서 빈 줄 없이 이어붙이면 마크다운 렌더러가 한 문단으로 합쳐버려 줄바꿈이 사라진다.)
            paragraphs = []
            if item.get("reason"):
                paragraphs.append(f"**사유:** {item['reason']}")

            evidence = item.get("evidence", [])
            if not evidence:
                paragraphs.append("**근거:** 문장 없음")
            for ev in evidence:
                pages = ev.get("pages") or []
                page_label = f"page {', '.join(str(p) for p in pages)}" if pages else "페이지 확인 불가"
                tag = "원문 확인됨" if ev.get("verified") else "⚠️ 원문 미확인(환각 의심)"
                paragraphs.append(f"**근거 ({page_label}, {tag}):**")
                paragraphs.append(render_quote_as_markdown(ev.get("quote", ""), ev.get("table_header")))

            lines.append(_blockquote("\n\n".join(paragraphs)))

    return "\n".join(lines)


# 예시 (5-2의 response 사용)
if PDF_PATHS:
    report = build_verification_report_markdown(response)
    print(report)

### 5-4. `objects`에 여러 개의 name이 들어올 때 — name별로 json/md 리포트 생성

`objects` 배열에 `name`이 다른 항목이 여러 개 들어오면, `name`별로 각각 독립된 응답 JSON과 마크다운 리포트를 만들어 파일로 저장합니다 (`data/verification_reports/<name>.json`, `<name>.md`). PDF는 5-1과 동일하게 1개 기준입니다 (여러 PDF와 대조하려면 PDF마다 한 번씩 호출). 마크다운 변환은 5-3의 `build_verification_report_markdown()`을 재사용합니다.

**`value`가 없는 item 처리:** `items`의 item에 `itemNm`만 있고 `value`가 없으면 (예: `REQUEST_SAMPLE`의 세 번째 item `{"itemNm": "이용 금액", "description": "설명"}`) 에러를 내는 대신 `verify_items_against_doc()`이 그 item을 건너뜁니다. `build_response()`/`build_and_save_reports_per_name()`도 같은 기준(`"value" in it`)으로 `items`를 미리 걸러서, `value`가 없는 item은 응답/리포트에도 포함되지 않습니다.

In [ ]:
import re as _re

REPORT_DIR = Path.cwd().parent / "data" / "verification_reports"


def _safe_filename(name: str) -> str:
    """name을 파일명으로 써도 안전하도록 OS에서 금지된 문자만 치환한다."""
    return _re.sub(r'[\\/:*?"<>|]', "_", name).strip() or "untitled"


def build_and_save_reports_per_name(request: dict, pdf_path) -> list[dict]:
    """request의 objects를 name별로 순회하며, name마다 독립된 응답을 만들어 json/md로 저장한다.
    value가 없는 item은 대조 대상이 아니므로 결과에서 제외한다."""
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    responses = []
    objects = request.get("objects", [])
    total = len(objects)
    for idx, obj in enumerate(objects, start=1):
        name = obj.get("name", "")
        print(f"[{idx}/{total}] 처리 중: {name}")
        items = [it for it in obj.get("items", []) if "value" in it]
        verified = verify_items_against_doc(name, items, pdf_path)
        result_by_key = {(r["itemNm"], r["value"]): r for r in verified["results"]}

        new_items = []
        for it in items:
            key = (it.get("itemNm"), it.get("value"))
            r = result_by_key.get(key, {})
            new_items.append(
                {
                    "itemNm": it.get("itemNm"),
                    "value": it.get("value"),
                    "verdict": r.get("verdict", "확인불가"),
                    "reason": r.get("reason", ""),
                    "evidence": [
                        {
                            "quote": e["quote"],
                            "verified": e["verified"],
                            "pages": e["pages"],
                            "table_header": e.get("table_header"),
                        }
                        for e in r.get("evidence", [])
                    ],
                }
            )

        response = {
            "request_id": request.get("request_id"),
            "doc": verified["doc"],
            "objects": [{"name": name, "items": new_items}],
        }

        filename = _safe_filename(name)
        json_path = REPORT_DIR / f"{filename}.json"
        md_path = REPORT_DIR / f"{filename}.md"
        json_path.write_text(json.dumps(response, ensure_ascii=False, indent=2), encoding="utf-8")
        md_path.write_text(build_verification_report_markdown(response), encoding="utf-8")

        responses.append(response)

    return responses


# 예시 (5-1의 REQUEST_SAMPLE, PDF_PATHS[0] 사용)
if PDF_PATHS:
    responses_per_name = build_and_save_reports_per_name(REQUEST_SAMPLE, PDF_PATHS[0])

In [ ]:
data = {"objects": [
        {
            "name": "요고 69",
            "items": [
                {"value": "Y", "itemNm": "초이스 상품여부", "desc":"itemNm 필드에 대한 설명" },
                {"value": "개인, 미성년자, 외국인", "itemNm": "이용 가능 고객", "desc":" itemNm 필드에 대한 설명"},
            ],
        }
    ]
}